Imports and path define 

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

source_path = "/Volumes/taxi_catalog/bronze/raw_files"

verify source path

In [0]:
display(dbutils.fs.ls(source_path))

Read all Parquet files

In [0]:
df_raw =spark.read.format("parquet").load(source_path)

display(df_raw.limit(20))

Inspect schema

In [0]:
df_raw.printSchema()
print("Total columns:", len(df_raw.columns))
print("Column names:", df_raw.columns)


Validate required columns

In [0]:
required_columns = [
    "VendorID",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "passenger_count",
    "trip_distance",
    "PULocationID",
    "DOLocationID",
    "fare_amount",
    "total_amount"
]

actual_columns = set(df_raw.columns)

missing_columns = [
    col for col in required_columns
    if col not in actual_columns
]

if missing_columns:
    raise ValueError(
        f"Schema validation failed. Missing columns: {missing_columns}"
    )


else:
    print("All required columns are present.")
    
print("Schema validation successful")

Add ingestion metadata


In [0]:
df_bronze = (
    df_raw
    .withColumn("source_file", F.col("_metadata.file_path"))
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("ingestion_date", F.current_date())
)

display(df_bronze.limit(10))

Write Bronze as Delta


In [0]:
df_bronze.write.format("delta").mode("overwrite").saveAsTable("taxi_catalog.bronze.yellow_taxi_raw")

Validate Bronze


In [0]:
display(spark.sql("select * from taxi_catalog.bronze.yellow_taxi_raw limit 19"))

In [0]:
df_check = spark.table("taxi_catalog.bronze.yellow_taxi_raw")

print("total_row", df_check.count())
print("total columns", len(df_check.columns))
print("column names", df_check.columns)
df_check.show()